In [75]:
import pyreadr
import pandas as pd
import re
from bs4 import BeautifulSoup
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, confusion_matrix
import numpy as np

example_preds = pyreadr.read_r('../../results/example-preds.RData')
example_preds_df = example_preds['pred_df']

#claims_test = pyreadr.read_r('../../data/claims-test.RData')
#claims_test_df = claims_test['claims_test']

claims_clean_ex = pyreadr.read_r('../../data/claims-clean-example.RData')
claims_clean_ex_df = claims_clean_ex['claims_clean']


raw_claims = pyreadr.read_r('../../data/claims-raw.RData')
raw_claims_df = raw_claims['claims_raw']

In [18]:
def parse_fn(html, selector = None):
    soup = BeautifulSoup(html, features = "lxml")
    try:
        for script in soup(["script", "style"]):
            script.decompose() # destroy script and style elements

        if selector is None:
            raw = ' '.join(soup.get_text(' '))
        else:
            raw = ' '.join(spoon.get_text(' ') for spoon in soup.select(selector))
        
        # remove urls and emails
            # \S+@\S+ means non-whitespace characters followed by @ then followed by 1 or more non-whitespace characters
            # pattern = r'https?://\S+|www\.\S+' which means http with optional s followed by :// and then 1 or more non-whitespace characters
            # |www\.\S+ means or www. then 1 or more non-whitespace characters
        text = re.sub(r'\S+@\S+|https?://\S+|www\.\S+', "", raw)

        # remove apostraphes

        text = re.sub("'", "",text)

        # remove non-alphabetic characters and nbsp
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text) # add space between camel case words

        text = text.lower()

        text = re.sub(r'\s+', ' ', text).strip() # collapse multiple spaces into one and strip leading/trailing whitespace
        # remove punctuation and newlines
        
        return text 
    except Exception as err:
        print(f"Error parsing HTML: {err}")
        return None


def parse_data(df, selector = None):
    df = df[df['text_tmp'].str.contains('<!', regex = False)].copy() # filter for rows that contain html tags
    df['text_clean'] = df['text_tmp'].apply(parse_fn, selector = selector) # apply parse_fn and create a new column

    print("Parse done!")
    return df



In [4]:
clean_claims_baseline = parse_data(raw_claims_df, 'p')
clean_claims_df = parse_data(raw_claims_df, selector = 'h1, h2, h3, h4, h5, h6, p')
# pyreadr.write_rdata("../../data/clean-claims-task1.RData", clean_claims_df, df_name = "dataset")

C:\Users\Felix\AppData\Local\Temp\ipykernel_60196\4244992399.py:2: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, features = "lxml")


Parse done!


C:\Users\Felix\AppData\Local\Temp\ipykernel_60196\4244992399.py:2: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, features = "lxml")


Parse done!


In [5]:

class SVDTxtPipeline:
    
    def __init__(self, prop = 0.7, ngram = None, max_components = 700, random_state = 101422):
        self.lemmatizer = WordNetLemmatizer()
        self.random_state = random_state
        self.prop = prop
        self.max_components = max_components
        # strip punctuation from stop words and lemmatize
        self.stpwrd = {w.replace("'", "") for w in ENGLISH_STOP_WORDS}
        self.tsvd = None
        self.n_pc_ = None
        self.cum_ = None
        # initialize Tfidf object
        if ngram == None or ngram.lower().strip() == "unigram":
            self.vectorizer = TfidfVectorizer(tokenizer = self.tokenize, lowercase = False, token_pattern = None)
        elif ngram.lower().strip() == "bigram":
            self.vectorizer = TfidfVectorizer(tokenizer = self.tokenize, ngram_range = (2,2), lowercase = False, token_pattern = None)

    def tokenize(self, text):
        # our tokens, we remove stopwords
        tokens = text.split()
        return [self.lemmatizer.lemmatize(token) for token in tokens if token not in self.stpwrd]
        
    def project(self, train_df):
        # create truncated svd object
        # Fit vectorizer and SVD on training data
        # create dtm with tfidf
        train_dtm = self.vectorizer.fit_transform(train_df['text_clean'])

        # prevent features less than obs
        k = min(self.max_components, train_dtm.shape[1] - 1)
        self.tsvd = TruncatedSVD(n_components = k, random_state = self.random_state)
        self.tsvd.fit(train_dtm)

        # explained variance ratio relative to the total variance
        # get cumsum to explain the prop of variance explained by components
        self.cum_ = self.tsvd.explained_variance_ratio_.cumsum()

        if self.cum_[-1] >= self.prop: # if cum variance exceeds the proportion 
            self.n_pc_ = int(np.argmax(self.cum_ >= self.prop) + 1) # get the first index where cum >= prop and add 1 to compensate for numpy 0-indexings

        else: # if the cum variance never exceeds the proportion
            self.n_pc_ = k # set to k
        

        return self

    def reproject(self, df):
        # transform test data without fit
        dtm = self.vectorizer.transform(df['text_clean'])

        # use transform to multiply the dtm by the first n_pc_ column (from fit)
        # reproject
        return self.tsvd.transform(dtm)[:, :self.n_pc_] 

        
    def process_train_test(self, train_df, test_df):
        # project training data
        self.project(train_df)

        x_train = self.reproject(train_df)
        x_test = self.reproject(test_df)

        y_train = (train_df['bclass'] == "Relevant claim content").astype(int) # convert to binary labels so that relevant = 1
        y_test = (test_df['bclass'] == "Relevant claim content").astype(int)

        return x_train, x_test, y_train, y_test

## Maybe learn to adapt for sklearn pipeline? -_- nah


In [89]:
def classification_metrics(model, x, y, dataset, model_wrapper):
    positive_column = np.flatnonzero(model.classes_ == 1)[0]
    probabilities = model.predict_proba(x)[:, positive_column]
    predictions = (probabilities > 0.5).astype(int)

    return {
        "dataset": dataset,
        "accuracy": accuracy_score(y, predictions),
        "roc_auc": roc_auc_score(y, probabilities),
        "sensitivity": recall_score(y, predictions, zero_division=0),
        "specificity": recall_score(
            y, predictions, pos_label=0, zero_division=0
        ),
        "cv accuracy" : model_wrapper["cv accuracy"],
        "cv accuracy sd" : model_wrapper["cv accuracy sd"],
        "cv roc auc" : model_wrapper["cv roc auc"],
        "cv roc auc sd" : model_wrapper["cv roc auc sd"]
    }


In [ ]:
def fitModel(model_name, df, is_meta = False, ngram = None, first_model = None):

    # Is bigram or more? Then create pipeline
    pipe = SVDTxtPipeline(ngram = ngram)

    # create train and test sets
    train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 101422)
    
    x_train, x_test, y_train, y_test = pipe.process_train_test(train_df, test_df)
    
    
    if is_meta: # is the model being run inside another model?
        if first_model is None:
            raise ValueError("is_meta = True requires first_model")

        try:
            # Take the x train and x test from first model
            # In our assignment it will be the unigram model
            first_x_train = first_model['x train']
            first_x_test = first_model['x test']

            # Principal components of first model is in first_x_train
            pos_col = np.flatnonzero(first_model['model'].classes_ == 1)[0]
            # get the probs from the training and test set of unigram model
            probs_train = first_model['model'].predict_proba(first_x_train)[:,pos_col]
            probs_test = first_model['model'].predict_proba(first_x_test)[:,pos_col]
            # calculate the unigram log-odds for train and test set
            logit_est_train = np.log(probs_train / (1- probs_train))
            logit_est_test = np.log(probs_test / (1- probs_test))

            # create the train and test set which combines the bigram pcs and the log-odds of the unigram model  
            x_train = np.column_stack((x_train, logit_est_train))
            x_test = np.column_stack((x_test, logit_est_test))


        except Exception as err:
            print(f"Something went wrong: {err}")
            return None
        
    model = LogisticRegression(max_iter = 1000)
        
        
    cv = StratifiedKFold(n_splits = 10, shuffle=True, random_state=101422)
    cv_results = cross_validate(model, x_train, y_train, cv  = cv,
                                scoring = ["accuracy", "roc_auc"])

    model.fit(x_train, y_train)
        
    return {
        "model name" : model_name,
        "model" : model,
        "x train" : x_train,
        "x test" : x_test,
        "y train" : y_train,
        "y test" : y_test,
        "pipe" : pipe,
        "cv accuracy" : cv_results["test_accuracy"].mean(),
        "cv accuracy sd" : cv_results["test_accuracy"].std(),
        "cv roc auc" : cv_results["test_roc_auc"].mean(),
        "cv roc auc sd" : cv_results["test_roc_auc"].std()
    }


In [ ]:
ph_fit = fitModel("Paragraphs and Headers", clean_claims_df)
base_fit = fitModel("Only Paragraphs", clean_claims_baseline)
bigram_fit = fitModel("Bigram model", clean_claims_baseline, ngram = 'bigram')
bipc_bilogit_fit = fitModel("Bigrams pcs with unigram log-odds", clean_claims_baseline, 
                            is_meta = True, ngram = 'bigram', first_model = ph_fit)

In [180]:
model_fits = {
    "Paragraphs and Headers" : ph_fit,
    "Paragraphs Only" : base_fit,
    "Paragraphs and Headers Bigrams" : bigram_fit,
    "Bigrams pcs with unigram log-odds" : bipc_bilogit_fit
}

In [102]:
ph_fit['pipe'].n_pc_

530

In [115]:
y_test = ph_fit['y test']
n = len(y_test)
print(n)

428


In [121]:
print(*[f"Model {m['model name']} num of principal components: {m['pipe'].n_pc_}\n" for m in model_fits.values()])

Model Paragraphs and Headers num of principal components: 530
 Model Only Paragraphs num of principal components: 437
 Model Bigram model num of principal components: 642
 Model Bigrams pcs with unigram log-odds num of principal components: 642



In [128]:
log_pc_metrics = pd.DataFrame([
    classification_metrics(model['model'], model['x test'], model['y test'], model['model name'], model) for model in model_fits.values()
]).sort_values(by = ["cv accuracy", "cv roc auc"], ascending = False).round(4)

In [153]:
l = log_pc_metrics.set_index('dataset')
gap_doc = (l.at["Only Paragraphs", "accuracy"] - l.at["Bigram model", "accuracy"]) * n
gap_doc

np.float64(24.010800000000017)

In [ ]:
def bootstrap_test(amodel, bmodel):
    model_a, model_b = amodel['model'], bmodel['model']
    x_test_a, x_test_b = amodel['x test'], bmodel['x test']
    y = amodel['y test'].to_numpy()

    # get predicitons
    pred_a = model_a.predict(x_test_a)
    pred_b = model_b.predict(x_test_b)
    # get which ones were right and wrong
    a_ok = pred_a == y
    b_ok = pred_b == y

    # get the raw scores
    s_a = model_a.decision_function(x_test_a)
    s_b = model_b.decision_function(x_test_b)

    # create lists to store differences
    diffs = []
    diffsROC = []
    # initiate randomstate
    rng = np.random.default_rng(101422)
    # 2000 iterations
    for _ in range(2000):
        i = rng.integers(low=0, high=len(y), size=len(y)) # create a series of random indices that correspond to rows of y
        if y[i].min() == y[i].max(): # avoid drawing with each label being identicial
            continue
        diffs.append(a_ok[i].mean() - b_ok[i].mean()) # compute difference each time then add the result to the list
        diffsROC.append(roc_auc_score(y[i], s_a[i]) - roc_auc_score(y[i], s_b[i]))

     # what is the interval from 2.5% perc and 97.5%?
    print("Accuracy bootstrap interval: ", np.percentile(diffs, [2.5, 97.5]))
    print("ROC AUC observed difference: ", (roc_auc_score(y, s_a) - roc_auc_score(y, s_b)))
    print("ROC AUC bootstrap interval: ", np.percentile(diffsROC, [2.5, 97.5]))
    # if it captures 0, it is not significant



In [179]:
bootstrap_test(bipc_bilogit_fit, base_fit)

Accuracy bootstrap interval:  [-0.03971963  0.0046729 ]
ROC AUC observed difference:  0.014193689267387177
ROC AUC bootstrap interval:  [0.00173567 0.02765204]


In [98]:
log_pc_metrics

,dataset,accuracy,roc_auc,sensitivity,specificity,cv accuracy,cv accuracy sd,cv roc auc,cv roc auc sd
3,Bigrams pcs with unigram log-odds,0.8364,0.8988,0.8498,0.8233,0.8697,0.0203,0.9338,0.0155
0,Paragraphs and Headers,0.8364,0.8941,0.8545,0.8186,0.8195,0.0250,0.8855,0.0201
1,Only Paragraphs,0.8528,0.8846,0.8592,0.8465,0.8166,0.0242,0.8760,0.0245
2,Bigram model,0.7967,0.8642,0.7981,0.7953,0.7921,0.0279,0.8474,0.0268


In [99]:
# display(log_pc_metrics.sort_values(by=["accuracy", "roc_auc"], ascending = False).round(4))
pyreadr.write_rdata("../../results/log_pc_metrics.RData", log_pc_metrics, df_name = "log_pc_metrics", compress="gzip")

In [93]:
pyreadr.read_r('../../results/log_pc_metrics.RData')

OrderedDict([('log_pc_metrics',
                                                  dataset  accuracy  roc_auc  sensitivity  \
              rownames                                                                      
              1                           Only Paragraphs    0.8528   0.8846       0.8592   
              3         Bigrams pcs with unigram log-odds    0.8364   0.8988       0.8498   
              0                    Paragraphs and Headers    0.8364   0.8941       0.8545   
              2                              Bigram model    0.7967   0.8642       0.7981   
              
                        specificity  
              rownames               
              1              0.8465  
              3              0.8233  
              0              0.8186  
              2              0.7953  )])